# Breast Cancer Prediction using Random Forest

**BSc final year project - Google Colab training notebook**

This notebook performs the complete machine learning workflow for the
Breast Cancer Prediction System: data loading, exploratory data
analysis, preprocessing, Random Forest training with hyperparameter
tuning, evaluation, model comparison, and saving the final artifacts
(`model.pkl`, `feature_names.pkl`, `metrics.pkl`) used by the
Streamlit application (`app.py`).

Dataset: Wisconsin Diagnostic Breast Cancer (WDBC), loaded directly
from `sklearn.datasets.load_breast_cancer()` - no external download
required.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 1. Load the dataset

We use scikit-learn's built-in copy of the WDBC dataset - no need to
download anything.

In [ ]:
dataset = load_breast_cancer()
X = pd.DataFrame(dataset.data, columns=dataset.feature_names)
y = pd.Series(dataset.target, name="target")  # 0 = malignant, 1 = benign

print(dataset.DESCR[:600])

In [ ]:
print("Feature names:", list(dataset.feature_names))
print("Target names:", list(dataset.target_names))
print("Shape:", X.shape)
X.head()

## 2. Exploratory data analysis (EDA)

We check for missing values, examine the class balance, visualize
correlations between features, and look at the distribution of a few
key measurements across the two diagnosis classes.

In [ ]:
print("Total missing values:", X.isnull().sum().sum())
print()
print("Class distribution:")
print(y.value_counts().rename(index=dict(enumerate(dataset.target_names))))

In [ ]:
X.describe().T[["mean", "std", "min", "max"]]

**Observation:** the dataset has no missing values, which simplifies
preprocessing considerably. The classes are moderately imbalanced
(more benign than malignant cases), which is why we use `stratify=y`
in the train/test split below and look at precision/recall/F1 in
addition to accuracy.

In [ ]:
plt.figure(figsize=(5, 4))
counts = y.value_counts().sort_index()
labels = [dataset.target_names[i] for i in counts.index]
sns.barplot(x=labels, y=counts.values, hue=labels, palette="viridis", legend=False)
plt.title("Class distribution (0 = malignant, 1 = benign)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.6})
plt.title("Feature correlation heatmap")
plt.show()

**Observation:** several features are highly correlated with each
other (e.g. radius, perimeter, and area, which are all measures of
size). Random Forest handles correlated features gracefully since
each tree only considers a random subset of features at each split,
so we do not need to manually remove correlated columns.

In [ ]:
key_features = ["mean radius", "mean texture", "mean smoothness", "mean concavity"]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
diagnosis = y.map(dict(enumerate(dataset.target_names)))
for ax, feat in zip(axes.ravel(), key_features):
    sns.histplot(data=X, x=feat, hue=diagnosis, kde=True, ax=ax)
    ax.set_title(feat)
plt.tight_layout()
plt.show()

In [ ]:
pair_df = X[key_features].copy()
pair_df["diagnosis"] = diagnosis
sns.pairplot(pair_df, hue="diagnosis", corner=True, palette="viridis")
plt.show()

## 3. Data preprocessing

We split the data into 80% training and 20% testing, using
`stratify=y` so both sets preserve the original class balance, and a
fixed `random_state=42` for reproducibility. Random Forest does not
require feature scaling (it splits on raw thresholds per feature), so
no scaler is applied.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 4. Model: Random Forest with hyperparameter tuning

We first measure a baseline Random Forest with default parameters
using 5-fold cross-validation, then search over a grid of
hyperparameters with `GridSearchCV` to find the best-performing
configuration.

In [ ]:
baseline = RandomForestClassifier(random_state=RANDOM_STATE)
baseline_scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="accuracy")
print(f"Baseline RF 5-fold CV accuracy: {baseline_scores.mean():.4f} (+/- {baseline_scores.std():.4f})")

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

## 5. Evaluation

We evaluate the tuned model on the held-out test set using accuracy,
precision, recall, F1-score, and ROC-AUC, along with a confusion
matrix, full classification report, ROC curve, and feature importance
chart.

- **Accuracy** - overall proportion of correct predictions.
- **Precision** - of all cases predicted malignant, how many actually were (avoids false alarms).
- **Recall (sensitivity)** - of all actually malignant cases, how many were caught (critical in a medical context - missing a malignant case is costly).
- **F1-score** - harmonic mean of precision and recall.
- **ROC-AUC** - the model's ability to separate the two classes across all thresholds.

In [ ]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1_score": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
}
for name, value in metrics.items():
    print(f"{name.replace('_', ' ').title()}: {value:.4f}")

In [ ]:
print(classification_report(y_test, y_pred, target_names=dataset.target_names))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=dataset.target_names, cmap="Blues", ax=ax
)
ax.set_title("Confusion matrix")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax)
ax.set_title("ROC curve")
plt.show()

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.values, y=importances.index, hue=importances.index, palette="viridis", legend=False)
plt.title("Top 15 feature importances (Random Forest)")
plt.xlabel("Importance")
plt.show()

**Observation:** measurements related to concavity, concave points,
and area/perimeter (particularly the "worst" variants) tend to be the
most important predictors, consistent with the medical intuition that
irregular, larger cell nuclei are associated with malignancy.

## 6. Model comparison

To justify choosing Random Forest as the final model, we train
baseline versions of four alternative classifiers on the same
train/test split and compare their metrics.

In [ ]:
candidates = {
    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=200),
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
}

comparison = {}
for name, clf in candidates.items():
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]
    comparison[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall": recall_score(y_test, preds),
        "f1_score": f1_score(y_test, preds),
        "roc_auc": roc_auc_score(y_test, proba),
    }

comparison_df = pd.DataFrame(comparison).T
comparison_df

In [ ]:
comparison_df.plot(kind="bar", figsize=(10, 6), colormap="viridis")
plt.title("Model comparison")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.ylim(0.8, 1.0)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Justification for choosing Random Forest:** across the five
candidate models, Random Forest and Logistic Regression are typically
the top two performers on this dataset. Random Forest is selected as
the final model because it (a) requires no feature scaling, (b)
provides built-in feature importance for interpretability - valuable
in a medical context, (c) is robust to the correlated features
observed in the EDA step, and (d) has lower variance than a single
Decision Tree due to ensembling, while remaining competitive with
Logistic Regression on recall for malignant cases.

## 7. Save the trained model and artifacts

We save the tuned Random Forest model, the ordered list of feature
names it expects, and the full metrics dictionary, so the Streamlit
application can load them directly without retraining.

In [ ]:
joblib.dump(best_model, "model.pkl")
joblib.dump(list(X.columns), "feature_names.pkl")

metrics["best_params"] = grid_search.best_params_
metrics["cv_best_score"] = grid_search.best_score_
metrics["classification_report"] = classification_report(y_test, y_pred, target_names=dataset.target_names)
metrics["confusion_matrix"] = confusion_matrix(y_test, y_pred).tolist()
metrics["feature_importance"] = importances.to_dict()
metrics["model_comparison"] = comparison
joblib.dump(metrics, "metrics.pkl")

print("Saved model.pkl, feature_names.pkl, and metrics.pkl")

## 8. Next steps

Download `model.pkl`, `feature_names.pkl`, and `metrics.pkl` from the
Colab file browser (or run this notebook locally / via the equivalent
`utils/train_model.py` script) and place them in the root of the
`Breast-Cancer-Prediction` project folder, alongside `app.py`. Then
run:

```bash
streamlit run app.py
```

to launch the full interactive prediction system.